# Modality specificity: native model weights
Full-dataset descriptive fit, using trial means within each condition and then the equal-weight average over conditions. Each point in a weight distribution is one native channel/source. No cross-modality channel pairing is used in these plots (coverage dataset construction still follows its selected definition).

We show **projection coefficients**, not activation/forward patterns. A large coefficient need not imply a large physiological response. All settings below are explicit; preprocessing follows `coverage_matching_utils`, as in `cov_models.ipynb`.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'coverage_matching_utils.py').exists():
    ROOT = ROOT / 'iEEGvsMEG'
for path in (ROOT, ROOT.parent, ROOT.parent / 'LB'):
    sys.path.insert(0, str(path))
from coverage_matching_utils import load_dataset
from cov_models_utils import fit_cov_models
from modality_specificity import threshold_weights, plot_weight_distributions, plot_thresholded_brains


In [ ]:
# Main selectors
MODEL = 'plssvd'  # 'plssvd', 'separate_pca', 'joint_pca'
MEG_KIND = 'full_concatenated'  # full_average, full_concatenated, coverage_average, paired_coverage, random_control
N_COMPONENTS = 5

MEG_DIR = ROOT / 'MEG' / 'dataMEG'
IEEG_DIR = ROOT / 'ieeg_shortWOBS_fs250'
SFREQ = 250.
CONDITIONS = (1, 2)
BLOCK_SCALING = 'none'  # same as cov_models; alternative: 'equal_variance'
MEG_SCALING = 'channel_zscore'
IEEG_MULTIPLIER = 1000.
MEG_COORDINATE_UNIT = 'm'  # unit in the source position CSVs; internal plotting uses mm
METADATA_CSV = None  # optional subject/channel_index/x/y/z table, MNI mm
SEED = 2026
OUTPUT = ROOT / 'out' / 'modality_specificity' / MEG_KIND / MODEL
assert MODEL in ('plssvd', 'separate_pca', 'joint_pca')
assert isinstance(N_COMPONENTS, int) and N_COMPONENTS > 0


## Load and fit
The iEEG features concatenate all subjects' channels. MEG uses the selected construction: concatenate native sources or average subjects as specified by `MEG_KIND`. The loader averages trials within condition; `condition_mode='average'` then averages conditions. No held-out data are used here.

Without `METADATA_CSV`, the loader uses the original `GetInfo` metadata and its existing coordinate correction. Explicit metadata is already in mm and is not corrected again. Adjust the directory, sampling rate and coordinate units to match your files.


In [ ]:
info_files = sorted(IEEG_DIR.glob('*_info.json'))
if not info_files:
    raise FileNotFoundError(f'No iEEG info files in {IEEG_DIR}')
first_info = json.loads(info_files[0].read_text())
ieeg = load_dataset('ieeg', meg_dir=MEG_DIR, ieeg_dir=IEEG_DIR,
    metadata_csv=METADATA_CSV, ieeg_coordinate_unit='mm',
    meg_coordinate_unit=MEG_COORDINATE_UNIT, meg_tmin=float(first_info['time_epoch'][0]),
    sfreq=SFREQ, conditions=CONDITIONS, condition_mode='average',
    meg_scaling=MEG_SCALING, ieeg_scaling='none', ieeg_multiplier=IEEG_MULTIPLIER)
meg = load_dataset(MEG_KIND, reference=ieeg, seed=SEED, condition_mode='average')
# Reuse the established fitting implementation and retain the selected model.
models = fit_cov_models(ieeg, meg, n_components=N_COMPONENTS, block_scaling=BLOCK_SCALING)
model = models[MODEL]
datasets = {'iEEG': ieeg, 'MEG': meg}
weights = {'iEEG': model.ieeg_weights[:, :N_COMPONENTS].copy(),
           'MEG': model.meg_weights[:, :N_COMPONENTS].copy()}
print({name: values.shape for name, values in weights.items()})


## Magnitudes and component-specific thresholds
For each modality and component separately, use $a_i=|w_i|$, $\tau=\operatorname{mean}(a)+\operatorname{std}(a)$ (population SD, `ddof=0`) and retain $a_i$ when $a_i\geq\tau$, otherwise set it to zero. The fitted model is unchanged. These are descriptive thresholds, not significance tests.

Density histograms handle different channel counts. Raw coefficient scales also depend on feature count, model normalization and block scaling; a higher absolute mean is not evidence that one modality contains more activity. Separate PCA components are independently ordered and are not matched between modalities.


In [ ]:
retained, summaries = {}, {}
for name, values in weights.items():
    retained[name], summaries[name] = threshold_weights(values)
summary = pd.concat([table.assign(modality=name) for name, table in summaries.items()], ignore_index=True)
display(summary)
fig = plot_weight_distributions(weights, summaries)
plt.show()


## Thresholded brain maps
Colours show retained **absolute projection weights**, with a separate scale per panel. Zeroed channels are hidden. Coordinates are MNI millimetres. For full concatenation, thresholding uses all feature rows independently. Exact co-locations are averaged **including zeroed rows only for rendering**, to avoid arbitrary marker overplotting; distributions and saved arrays keep every original feature. No nearest-neighbour matching or anatomical-region aggregation is applied.


In [ ]:
brain_fig = plot_thresholded_brains(datasets, retained)
plt.show()


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT / 'thresholds.csv', index=False)
for name in weights:
    np.savez_compressed(OUTPUT / f'{name.lower()}_weights.npz',
        weights=weights[name], magnitude=np.abs(weights[name]), thresholded_magnitude=retained[name],
        coordinates_mm=datasets[name].metadata[['x', 'y', 'z']].to_numpy())
    datasets[name].metadata.to_csv(OUTPUT / f'{name.lower()}_features.csv', index=False)
fig.savefig(OUTPUT / 'weight_distributions.png', dpi=180, bbox_inches='tight')
brain_fig.savefig(OUTPUT / 'thresholded_brains.png', dpi=180, bbox_inches='tight')
(OUTPUT / 'configuration.json').write_text(json.dumps(dict(model=MODEL, meg_kind=MEG_KIND,
    n_components=N_COMPONENTS, block_scaling=BLOCK_SCALING, meg_scaling=MEG_SCALING,
    ieeg_multiplier=IEEG_MULTIPLIER, conditions=CONDITIONS, seed=SEED,
    meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR), sfreq=SFREQ,
    meg_coordinate_unit=MEG_COORDINATE_UNIT, metadata_csv=str(METADATA_CSV)), indent=2))
print(OUTPUT)


## Spatial organization of the group-model weights
Use **signed, unthresholded** first-k weight profiles from the model fitted above. No new PCA/PLSSVD fit is performed. For every distinct unordered pair:

$$d_{ij}=\|r_i-r_j\|_2,\qquad D_{ij}=1-\frac{w_i^T w_j}{\|w_i\|\|w_j\|}.$$

Cosine distance is 0 for identical directions, 1 for orthogonal profiles, and 2 for opposite directions. It ignores overall profile magnitude and is unchanged by a simultaneous sign flip of a component. With only one component it measures sign agreement alone. Zero-norm profiles are excluded and recorded in the channel audit.

The **group** curve uses all pairs, including pairs across subjects and co-located features (distance zero). Every pair has equal weight, so subjects with more channels contribute more. The **subject** curves restrict pairs to each subject without refitting the group model. Subject-averaged MEG has only a `participant_average` cloud; this cannot recover individual subjects' weights.

**Null:** shuffle coordinates within each subject independently, keeping the complete k-component profile together. Computationally, permuting profiles against fixed coordinates is equivalent. Group and subject results use the same permutations. This preserves subject-specific coordinate coverage and weight distributions. The gray band is the pointwise 2.5–97.5% permutation envelope, not a confidence interval. Two-sided Monte Carlo p-values use both permutation tails and a +1 correction; Holm correction covers all populated group/subject bins within each modality. The two modalities are separate test families. These tests assess spatial association, not whether MEG and iEEG differ, and do not remove MEG source-reconstruction smoothness.

All channel pairs are computed exactly in blocks, without subsampling. Runtime scales quadratically with channel count and linearly with permutations. Full-concatenated MEG can therefore be slow. Results are saved after each modality and can be loaded without recomputing. Bins are left-closed/right-open, except the final bin includes its right edge. Empty bins remain NaN.


In [ ]:
from modality_specificity import spatial_weight_distance, plot_spatial_weight_distance
DISTANCE_BINS_MM = np.arange(0, 310, 10)
N_SPATIAL_PERMUTATIONS = 199  # increase for finer p-value resolution, especially with Holm correction
SPATIAL_BLOCK_SIZE = 512
SPATIAL_SEED = 2026
RECOMPUTE_SPATIAL = True  # False loads previously saved matching results
SPATIAL_OUTPUT = OUTPUT / 'spatial_distance'
SPATIAL_OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import hashlib
from modality_specificity import spatial_metadata
spatial_results = {}
for modality, dataset in datasets.items():
    metadata = spatial_metadata(dataset.metadata)
    folder = SPATIAL_OUTPUT / modality.lower()
    folder.mkdir(parents=True, exist_ok=True)
    # Guard against loading results from a different fit, row ordering or configuration.
    digest = hashlib.sha256()
    digest.update(np.ascontiguousarray(weights[modality], dtype=np.float64).tobytes())
    digest.update(metadata[['subject', 'x', 'y', 'z']].to_csv(index=False).encode())
    config = dict(fingerprint=digest.hexdigest(), bins=DISTANCE_BINS_MM.tolist(),
                  permutations=N_SPATIAL_PERMUTATIONS, seed=SPATIAL_SEED,
                  model=MODEL, meg_kind=MEG_KIND, n_components=N_COMPONENTS,
                  null='within_subject_coordinate_shuffle', signed=True, version=1)
    if RECOMPUTE_SPATIAL:
        print(f'{modality}: exact all-pair analysis of {len(weights[modality])} channels', flush=True)
        result = spatial_weight_distance(weights[modality], metadata,
            bin_edges=DISTANCE_BINS_MM, n_permutations=N_SPATIAL_PERMUTATIONS,
            seed=SPATIAL_SEED, block_size=SPATIAL_BLOCK_SIZE, progress=print)
        for name, table in result.items():
            table.to_csv(folder / f'{name}.csv', index=False)
        (folder / 'configuration.json').write_text(json.dumps(config, indent=2))
    else:
        if json.loads((folder / 'configuration.json').read_text()) != config:
            raise ValueError(f'{modality}: saved configuration/weights differ; recompute.')
        result = {name: pd.read_csv(folder / f'{name}.csv', dtype={'subject': str})
                  for name in ('curves', 'null_curves', 'channel_audit')}
    spatial_results[modality] = result
    display(result['channel_audit'])
    display(result['curves'].query("scope == 'group'"))


In [ ]:
# Re-run only this cell to redraw already computed results.
for modality, result in spatial_results.items():
    for index, spatial_fig in enumerate(plot_spatial_weight_distance(result, title=f'{MODEL} · {modality}')):
        spatial_fig.savefig(SPATIAL_OUTPUT / modality.lower() / f'distance_curves_{index}.png',
                            dpi=180, bbox_inches='tight')
        display(spatial_fig)
        plt.close(spatial_fig)


## Individual components: normalized semivariograms
Use signed **unthresholded** weights from the same group fit. For component c and pairs in a distance bin, compute the mean of $(w_{ic}-w_{jc})^2/(2s_c^2)$. The variance $s_c^2$ is computed across all channels for the group curve and within each subject for its curve (population variance). Constant components within a scope have undefined normalized semivariance and remain NaN; zero weights are otherwise retained. Low values mean similar signed weights. A global component sign flip leaves this statistic unchanged.

As above, use all unordered pairs and shuffle coordinates within subjects. Null bands are pointwise permutation envelopes. Holm correction spans all populated bins, components and group/subject scopes **within each modality**. With 199 permutations, corrected significance can be difficult to resolve; increase permutations for inferential runs. Runtime scales additionally with the number of components. Results are saved and can be reloaded.


In [ ]:
from modality_specificity import (component_semivariograms, plot_component_semivariograms,
                                  weight_clusters, plot_weight_cluster_centroids)
RECOMPUTE_COMPONENT_SPATIAL = True
CLUSTER_RADIUS_MM = 10.  # distance-based adjacency; inspect sensitivity to this choice
CLUSTER_MIN_SOURCES = 3
COMPONENT_OUTPUT = OUTPUT / 'component_spatial'
COMPONENT_OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
component_spatial = {}
for modality, dataset in datasets.items():
    metadata = spatial_metadata(dataset.metadata)
    folder = COMPONENT_OUTPUT / modality.lower()
    folder.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    digest.update(np.ascontiguousarray(weights[modality], dtype=np.float64).tobytes())
    digest.update(metadata[['subject', 'x', 'y', 'z']].to_csv(index=False).encode())
    config = dict(fingerprint=digest.hexdigest(), bins=DISTANCE_BINS_MM.tolist(),
                  permutations=N_SPATIAL_PERMUTATIONS, seed=SPATIAL_SEED,
                  statistic='normalized_semivariance', version=1)
    if RECOMPUTE_COMPONENT_SPATIAL:
        result = component_semivariograms(weights[modality], metadata,
            bin_edges=DISTANCE_BINS_MM, n_permutations=N_SPATIAL_PERMUTATIONS,
            seed=SPATIAL_SEED, block_size=SPATIAL_BLOCK_SIZE, progress=print)
        for name, table in result.items(): table.to_csv(folder / f'{name}.csv', index=False)
        (folder / 'configuration.json').write_text(json.dumps(config, indent=2))
    else:
        if json.loads((folder / 'configuration.json').read_text()) != config:
            raise ValueError('Saved component configuration differs; recompute.')
        result = {name: pd.read_csv(folder / f'{name}.csv', dtype={'subject': str})
                  for name in ('curves', 'null_curves', 'channel_audit')}
    component_spatial[modality] = result


In [ ]:
for modality, result in component_spatial.items():
    for index, figure in enumerate(plot_component_semivariograms(result, f'{MODEL} · {modality}')):
        figure.savefig(COMPONENT_OUTPUT / modality.lower() / f'semivariogram_{index}.png', dpi=180, bbox_inches='tight')
        display(figure)
        plt.close(figure)


## Spatially connected clusters of model weights
For **each subject and component**, select $|w|\geq\operatorname{mean}(|w|)+\operatorname{SD}(|w|)$ using that subject's rows of the group-model weights. This is a new subject-specific threshold, distinct from the modality-wide threshold above. Positive and negative selections form separate graphs; left/right hemispheres (MNI x sign) and exact-midline sources are also separate. Link sources within `CLUSTER_RADIUS_MM` and retain connected groups of at least `CLUSTER_MIN_SOURCES`. The radius limits individual links, not the overall cluster diameter. Exact co-located rows remain included; the table also reports unique-location counts.

The centroid is $\sum_i |w_i|r_i/\sum_i |w_i|$. Report weighted RMS spatial spread, source count, total magnitude and the nearest **member source** to this centroid. Brain plots mark these real sources (red positive / blue negative) across subjects; exact centroid coordinates and per-feature membership are in saved CSVs. No cross-subject centroid matching is performed, so overlapping markers need not represent one cluster. Subject-averaged MEG yields only a group-average cloud.

These are descriptive clusters of model weights, not identified neural generators. MNI-distance adjacency can connect across cortical folds; source-mesh adjacency would be preferable if available. Inspect stability across radius and threshold choices before interpreting centroids biologically. Leakage and neural activity cannot be separated by these maps alone. Clusters use thresholds, but semivariograms never do.


In [ ]:
cluster_results = {}
for modality, dataset in datasets.items():
    result = weight_clusters(weights[modality], dataset.metadata,
        radius_mm=CLUSTER_RADIUS_MM, min_sources=CLUSTER_MIN_SOURCES)
    cluster_results[modality] = result
    folder = COMPONENT_OUTPUT / modality.lower() / 'clusters'
    folder.mkdir(parents=True, exist_ok=True)
    for name, table in result.items(): table.to_csv(folder / f'{name}.csv', index=False)
    (folder / 'configuration.json').write_text(json.dumps(dict(model=MODEL, meg_kind=MEG_KIND,
        n_components=N_COMPONENTS, radius_mm=CLUSTER_RADIUS_MM, min_sources=CLUSTER_MIN_SOURCES,
        threshold='within_subject_component_abs_mean_plus_sd'), indent=2))
    display(result['clusters'])
    for index, figure in enumerate(plot_weight_cluster_centroids(result, f'{MODEL} · {modality}')):
        figure.savefig(folder / f'centroids_component_{index+1}.png', dpi=180, bbox_inches='tight')
        display(figure)
        plt.close(figure)
